# CNN Model Training & Evaluation
## Module 3 - Computer Vision Learning

**⚠️ Run this notebook on Google Colab with T4 GPU enabled**

This notebook handles:
1. Build CNN model
2. Train with early stopping and checkpoints
3. Evaluate model
4. Plot loss and accuracy
5. Test with real data

## Setup & Imports

In [ ]:
# Check GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"GPU Available: {torch.cuda.is_available()}")
print(f"GPU Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

if not torch.cuda.is_available():
    print("\n⚠️ WARNING: No GPU found! Please enable GPU in Colab:")
    print("   Runtime → Change runtime type → GPU (T4 recommended)")

In [2]:
# Mount Google Drive (only for Colab)
from google.colab import drive
drive.mount('/content/drive')
print("✓ Google Drive mounted")

Mounted at /content/drive
✓ Google Drive mounted


In [ ]:
import os
import shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from PIL import Image

# Metrics
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Set random seeds
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print("✓ All imports successful")

In [4]:
# Define paths - adjust if dataset is in Google Drive
# Example for Google Drive:
DATASET_BASE = Path('/content/drive/MyDrive/Data Science/Computer Vision/learning-assignments/datasets/flowers')  # Change this path
DATASET_SPLITS = DATASET_BASE / 'splits'
TRAIN_DIR = DATASET_SPLITS / 'train'
VAL_DIR = DATASET_SPLITS / 'val'
TEST_DIR = DATASET_SPLITS / 'test'

# Create output directory
OUTPUT_DIR = Path('/content/drive/MyDrive/Data Science/Computer Vision/learning-assignments/flower_classifier_output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Train directory: {TRAIN_DIR}")
print(f"Train directory exists: {TRAIN_DIR.exists()}")
print(f"Output directory: {OUTPUT_DIR}")

Train directory: /content/drive/MyDrive/Data Science/Computer Vision/learning-assignments/datasets/flowers/splits/train
Train directory exists: True
Output directory: /content/drive/MyDrive/Data Science/Computer Vision/learning-assignments/flower_classifier_output


## Step 1: Data Loading & Preprocessing

In [ ]:
# Image dimensions and batch size
IMG_HEIGHT = 224
IMG_WIDTH = 224
BATCH_SIZE = 32
NUM_CLASSES = 3

# Data transforms - NO AUGMENTATION (normalize only)
transform = transforms.Compose([
    transforms.Resize((IMG_HEIGHT, IMG_WIDTH)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

print("Data transforms configured (no augmentation)")

In [ ]:
# Load datasets
train_dataset = ImageFolder(root=TRAIN_DIR, transform=transform)
val_dataset = ImageFolder(root=VAL_DIR, transform=transform)
test_dataset = ImageFolder(root=TEST_DIR, transform=transform)

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# Get class names
class_names = train_dataset.classes
class_names.sort()

print(f"\nClass names: {class_names}")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

In [ ]:
# Visualize sample images
fig, axes = plt.subplots(3, 3, figsize=(10, 10))
axes = axes.ravel()

# Get a batch of images
images_batch, labels_batch = next(iter(train_loader))

# Denormalize and plot (undo normalization for visualization)
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

for i in range(min(9, len(images_batch))):
    # Denormalize
    img = images_batch[i].numpy().transpose((1, 2, 0))
    img = std * img + mean
    img = np.clip(img, 0, 1)
    
    axes[i].imshow(img)
    class_idx = labels_batch[i].item()
    axes[i].set_title(class_names[class_idx], fontweight='bold')
    axes[i].axis('off')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'sample_images.png', dpi=100, bbox_inches='tight')
plt.show()

print("✓ Sample images visualization saved")

## Step 2: Build CNN Model

In [ ]:
class FlowerCNN(nn.Module):
    def __init__(self, num_classes):
        super(FlowerCNN, self).__init__()
        
        # Block 1
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.25)
        )
        
        # Block 2
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.25)
        )
        
        # Block 3
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.25)
        )
        
        # Block 4
        self.block4 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.25)
        )
        
        # Fully connected layers
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 14 * 14, 512),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(512),
            nn.Dropout(0.5),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(256),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.fc(x)
        return x

# Build model
model = FlowerCNN(NUM_CLASSES).to(device)

# Display model architecture
print("\n" + "="*60)
print("MODEL ARCHITECTURE")
print("="*60)
print(model)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

In [ ]:
# Setup optimizer, loss function, and learning rate scheduler
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-6, verbose=True
)

print("✓ Optimizer and loss function configured")

## Step 3: Train Model with Early Stopping & Checkpoints

In [ ]:
# Setup checkpointing directory
checkpoint_dir = OUTPUT_DIR / 'checkpoints'
checkpoint_dir.mkdir(parents=True, exist_ok=True)

# Early stopping configuration
early_stopping_patience = 15
best_val_loss = float('inf')
patience_counter = 0
best_val_acc = 0.0

print(f"Checkpoint directory: {checkpoint_dir}")
print("✓ Training configuration ready")

In [ ]:
# Training loop
def train_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Statistics
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

def val_epoch(model, val_loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    epoch_loss = running_loss / len(val_loader)
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

# Training
print("\n" + "="*60)
print("STARTING MODEL TRAINING")
print("="*60)
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

EPOCHS = 100
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = val_epoch(model, val_loader, criterion, device)
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    # Learning rate scheduling
    scheduler.step(val_loss)
    
    # Early stopping and checkpointing
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_val_loss = val_loss
        patience_counter = 0
        # Save checkpoint
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'val_loss': val_loss,
        }, checkpoint_dir / 'best_model.pt')
    else:
        patience_counter += 1
    
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/{EPOCHS} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    
    if patience_counter >= early_stopping_patience:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break

print(f"\nEnd time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("✓ Training completed")

## Step 4: Plot Loss & Accuracy

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy plot
axes[0].plot(history['train_acc'], label='Train Accuracy', linewidth=2)
axes[0].plot(history['val_acc'], label='Val Accuracy', linewidth=2)
axes[0].set_title('Model Accuracy', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Loss plot
axes[1].plot(history['train_loss'], label='Train Loss', linewidth=2)
axes[1].plot(history['val_loss'], label='Val Loss', linewidth=2)
axes[1].set_title('Model Loss', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_history.png', dpi=100, bbox_inches='tight')
plt.show()

print("✓ Training history plots saved")

In [ ]:
# Print final metrics
print("\n" + "="*60)
print("TRAINING METRICS")
print("="*60)

final_train_acc = history['train_acc'][-1]
final_val_acc = history['val_acc'][-1]
final_train_loss = history['train_loss'][-1]
final_val_loss = history['val_loss'][-1]

print(f"\nFinal Epoch Metrics:")
print(f"  Train Accuracy: {final_train_acc:.4f}")
print(f"  Val Accuracy:   {final_val_acc:.4f}")
print(f"  Train Loss:     {final_train_loss:.4f}")
print(f"  Val Loss:       {final_val_loss:.4f}")

best_epoch = history['val_acc'].index(max(history['val_acc'])) + 1

print(f"\nBest Val Accuracy: {best_val_acc:.4f} (Epoch {best_epoch})")

## Step 5: Evaluate Model on Test Set

In [ ]:
# Load best model
best_model = FlowerCNN(NUM_CLASSES).to(device)
checkpoint = torch.load(checkpoint_dir / 'best_model.pt')
best_model.load_state_dict(checkpoint['model_state_dict'])
best_model.eval()
print("✓ Best model loaded from checkpoint")

In [ ]:
# Evaluate on test set
print("\n" + "="*60)
print("TEST SET EVALUATION")
print("="*60)

best_model.eval()
test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = best_model(images)
        loss = criterion(outputs, labels)
        
        test_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

test_loss = test_loss / len(test_loader)
test_accuracy = correct / total

print(f"\nTest Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

In [ ]:
# Get predictions on test set
predicted_classes = []
true_classes = []

best_model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = best_model(images)
        _, predicted = torch.max(outputs, 1)
        predicted_classes.extend(predicted.cpu().numpy())
        true_classes.extend(labels.numpy())

predicted_classes = np.array(predicted_classes)
true_classes = np.array(true_classes)

print("✓ Predictions generated")

In [ ]:
# Classification report
print("\n" + "="*60)
print("CLASSIFICATION REPORT")
print("="*60)

report = classification_report(
    true_classes,
    predicted_classes,
    target_names=class_names,
    digits=4
)

print(report)

In [ ]:
# Confusion matrix
cm = confusion_matrix(true_classes, predicted_classes)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names,
    cbar_kws={'label': 'Count'}
)
ax.set_title('Confusion Matrix - Test Set', fontsize=12, fontweight='bold')
ax.set_ylabel('True Label')
ax.set_xlabel('Predicted Label')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'confusion_matrix.png', dpi=100, bbox_inches='tight')
plt.show()

print("✓ Confusion matrix saved")

## Step 6: Test with Real Data

In [ ]:
def predict_single_image(model, image_path, class_names, device):
    model.eval()
    with torch.no_grad():
        # Load and preprocess image
        img = Image.open(image_path).convert('RGB')
        img_tensor = transform(img).unsqueeze(0).to(device)
        
        # Predict
        output = model(img_tensor)
        probs = torch.softmax(output, dim=1)
        predicted_class_idx = torch.argmax(probs[0]).item()
        confidence = probs[0][predicted_class_idx].item()
        
        return class_names[predicted_class_idx], confidence, probs[0].cpu().numpy()

print("✓ Prediction function ready")

In [ ]:
# Test on random samples from test set
test_sample_dir = TEST_DIR
all_test_images = []

for class_dir in test_sample_dir.iterdir():
    if class_dir.is_dir():
        for img_file in class_dir.glob('*.jpg'):
            all_test_images.append((img_file, class_dir.name))
        for img_file in class_dir.glob('*.jpeg'):
            all_test_images.append((img_file, class_dir.name))
        for img_file in class_dir.glob('*.png'):
            all_test_images.append((img_file, class_dir.name))

# Select random samples
sample_indices = np.random.choice(len(all_test_images), size=min(9, len(all_test_images)), replace=False)
sample_images = [all_test_images[i] for i in sample_indices]

print(f"\n" + "="*60)
print("TEST PREDICTIONS ON RANDOM SAMPLES")
print("="*60)

fig, axes = plt.subplots(3, 3, figsize=(12, 12))
axes = axes.ravel()

for idx, (img_path, true_class) in enumerate(sample_images):
    # Predict
    pred_class, confidence, all_probs = predict_single_image(best_model, img_path, class_names, device)

    # Load and display image
    img = Image.open(img_path)
    axes[idx].imshow(img)

    # Title with prediction and true label
    is_correct = pred_class == true_class
    symbol = '✓' if is_correct else '✗'
    title = f"{symbol} Pred: {pred_class}\nTrue: {true_class}\nConf: {confidence:.2%}"
    color = 'green' if is_correct else 'red'
    axes[idx].set_title(title, fontweight='bold', color=color)
    axes[idx].axis('off')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'test_predictions.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n✓ Test predictions visualization saved")

In [ ]:
def test_image_interactive(image_path_str):
    image_path = Path(image_path_str)

    if not image_path.exists():
        print(f"Error: File not found - {image_path}")
        return

    pred_class, confidence, all_probs = predict_single_image(best_model, image_path, class_names, device)

    print(f"\n{'='*50}")
    print(f"Image: {image_path.name}")
    print(f"{'='*50}")
    print(f"\nPredicted Class: {pred_class}")
    print(f"Confidence: {confidence:.2%}")
    print(f"\nClass Probabilities:")
    for class_name, prob in zip(class_names, all_probs):
        bar = '█' * int(prob * 50)
        print(f"  {class_name:12s}: {prob:.4f} {bar}")

    # Display image
    img = Image.open(image_path)
    plt.figure(figsize=(6, 6))
    plt.imshow(img)
    plt.title(f"Predicted: {pred_class} ({confidence:.2%})", fontsize=12, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

print("\n✓ Interactive testing function ready")
print("  Usage: test_image_interactive('/path/to/image.jpg')")

## Save Model & Summary

In [ ]:
# Save final model
model_path = OUTPUT_DIR / 'flower_classifier_model.pt'
torch.save(best_model.state_dict(), model_path)
print(f"✓ PyTorch model saved: {model_path}")

# Save as ONNX format (for production deployment)
try:
    onnx_path = OUTPUT_DIR / 'flower_classifier_model.onnx'
    dummy_input = torch.randn(1, 3, IMG_HEIGHT, IMG_WIDTH).to(device)
    torch.onnx.export(best_model, dummy_input, onnx_path, 
                      input_names=['input'], output_names=['output'],
                      verbose=False)
    print(f"✓ ONNX model saved: {onnx_path}")
except Exception as e:
    print(f"Note: ONNX export skipped ({e})")

In [ ]:
# Create summary report
total_params = sum(p.numel() for p in best_model.parameters())

summary_report = f"""
FLOWER CLASSIFICATION MODEL - TRAINING SUMMARY
{'='*60}

MODEL ARCHITECTURE:
  Framework: PyTorch
  Type: CNN (Convolutional Neural Network)
  Total Parameters: {total_params:,}
  Input Shape: {IMG_HEIGHT}x{IMG_WIDTH}x3
  Classes: {NUM_CLASSES} ({', '.join(class_names)})

TRAINING CONFIGURATION:
  Epochs (trained): {len(history['train_loss'])}
  Batch Size: {BATCH_SIZE}
  Optimizer: Adam (lr=1e-3)
  Loss Function: CrossEntropyLoss
  Data Augmentation: DISABLED (baseline testing)
  Early Stopping Patience: {early_stopping_patience}

TRAINING RESULTS:
  Best Val Accuracy: {best_val_acc:.4f} (Epoch {best_epoch})
  Final Train Accuracy: {final_train_acc:.4f}
  Final Val Accuracy: {final_val_acc:.4f}
  Final Train Loss: {final_train_loss:.4f}
  Final Val Loss: {final_val_loss:.4f}

TEST SET PERFORMANCE:
  Test Accuracy: {test_accuracy:.4f}
  Test Loss: {test_loss:.4f}

OUTPUTS:
  ✓ Model: flower_classifier_model.pt
  ✓ ONNX Model: flower_classifier_model.onnx
  ✓ Training History: training_history.png
  ✓ Confusion Matrix: confusion_matrix.png
  ✓ Test Predictions: test_predictions.png
  ✓ Sample Images: sample_images.png
  ✓ Checkpoints: checkpoints/best_model.pt

GENERATED: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
"""

print(summary_report)

# Save report
with open(OUTPUT_DIR / 'training_summary.txt', 'w') as f:
    f.write(summary_report)

print(f"\n✓ Summary saved: {OUTPUT_DIR / 'training_summary.txt'}")

In [ ]:
print("\n" + "="*60)
print("✓ TRAINING AND EVALUATION COMPLETE")
print("="*60)
print(f"\nAll outputs saved to: {OUTPUT_DIR}")
print(f"\nYou can now:")
print(f"  1. Download the model for deployment")
print(f"  2. Review the training history and metrics")
print(f"  3. Use test_image_interactive() to test on custom images")
print(f"\nNext steps:")
print(f"  - Try with data augmentation by modifying the transform")
print(f"  - Fine-tune hyperparameters for better accuracy")
print(f"  - Deploy model using the .pt or .onnx file")